# 01 - Loading and cleaning the raw FriendFeed data

| Raw file | Rows | Problems found | Output |
|---|---|---|---|
| users.csv | 671,840 | 46,158 groups (not people); description missing as text `'null'` | `users.parquet` |
| services.csv | 1,587,347 | 72 malformed lines ("\|" inside values) skipped; 380,599 duplicate user–service rows | `services.parquet` (1,206,676) |
| entries1–3.csv | 12,450,658 | GeoX/GeoY 100% empty; 13,282 posts dated before the window (incl. 1970) | `entries.parquet` |
| commentAugSept.csv | 3,749,891 | 9 columns, not 13 (README wrong); 130,995 before the window; 217 orphans | `comments.parquet` |
| likes.csv | 798,112 | 822 duplicates; dates 2008 – Oct 2010; 17.4% orphans (0.1% in window) | `likes.parquet` (797,290) |
| subscriptions.csv | 27,811,816 | Not in README; column order opposite to README; 1,948,989 duplicate edges | `follows.parquet` (25,862,827) |
| followingAugSept.csv | 19,547,158 | Timestamps = crawl time, not follow time | **not used** |

**Scope:** no text content is used (agreed with the teacher). Only counts, lengths and yes/no flags are kept.
**Next step:** restrict all activity to the study window (1 Aug – 30 Sep 2010) and aggregate to one row per user.


In [ ]:
from pathlib import Path
import duckdb

ROOT = Path.cwd().parent                      # the Project Folder
RAW = ROOT / "Data Source_Friendfeed study"   # raw CSV files
OUT = ROOT / "data"                           # cleaned files go here

con = duckdb.connect()                        # an in-memory database

con.execute("SET memory_limit = '8GB'")        # Leaving RAM for windows, VS Code, the browser etc
con.execute(f"SET temp_directory = '{(ROOT / 'duckdb_tmp').as_posix()}'")        # This is where the spilled files go when there is any

print(Path.cwd())
print(RAW.exists(), OUT.exists())



c:\Users\HP\Downloads\Data Mining I\Project\Code
True True


Looking the Raw files before loading them

In [4]:
files = ["users.csv", "services.csv", "subscriptions.csv", "likes.csv", "entries1.csv", "commentAugSept.csv", "followingAugSept.csv"]

for name in files:
    print("====", name, "====")

    with open(RAW / name, "r", encoding="utf-8", errors="replace") as f:
        for _ in range(5):
            line = f.readline()
            print(repr(line[:150]))
    print()

==== users.csv ====
'2gwsk|user|Ming Quek|2a9209ad|null\n'
'iroach|user|Isaac Roach|20948a19|null\n'
'tonybraun|user|Tony Braun|f68e8b26|null\n'
'junknajidyahoocom1|user|Junknajid Yahoocom|98a02bb6|null\n'
'spaceastronautics|user|Space & Astronautics News|c3178cfa|NASA, ESA, NOAA, USGS, astronomy, space, weather, hurricanes, oceans, Earth science, animals'

==== services.csv ====
'2gwsk|blog|Blog|http://en.wikipedia.org/wiki/Blog|null|http://2gwsk.wordpress.com\n'
'2gwsk|twitter|Twitter|http://twitter.com/|2gwsk|http://twitter.com/2gwsk\n'
'iroach|facebook|Facebook|http://www.facebook.com/|null|http://www.facebook.com/profile.php?id=688384411\n'
'iroach|googletalk|Gmail/Google Talk|http://talk.google.com/|null|null\n'
'iroach|twitter|Twitter|http://twitter.com/|iroach|http://twitter.com/iroach\n'

==== subscriptions.csv ====
'seoptimise,2gwsk\n'
'xahac,2gwsk\n'
'kmorrison,iroach\n'
'seoptimise,iroach\n'
'skyje,iroach\n'

==== likes.csv ====
'yasinde\te/2d4fcaed2d2a4ac1a36571b66da3f16c\

| File | Separator | Header row? | Missing values look like | Number of columns |
|------|-----------|-------------|--------------------------|-------------------|
| users.csv | `\|` (pipe) | No | the text `null` | 5 |
| services.csv | `\|` (pipe) | No | the text `null` | 6 |
| subscriptions.csv | `,` (comma) | No | none seen | 2 |
| followingAugSept.csv | `\t` (tab) | No | none seen | 3 |
| likes.csv | `\t` (tab) | No | none seen | 3 |
| entries1.csv | `\t` (tab) | No | `\N`, or empty (nothing between two tabs) | 12 |
| commentAugSept.csv | `\t` (tab) | No | `\N`, or empty | 9 (README says 13!) |


### Findings – raw file formats
- None of the files has a header row - column names were taken from the README.
- Files use three different separators (`|`, `,`, tab) and three ways of marking a missing value (`null`, `\N`, empty field) - each file needs its own read settings.
- **Documentation ≠ data:** the README lists 13 fields for comments, but `commentAugSept.csv` has only 9 (no image/video columns).
- An empty `SourceName` in entries is not really missing: it means the post was written directly on FriendFeed (a *native* post).
- `likes.csv` contains dates from March/April 2010, outside the Aug–Sep study window - must be filtered later.
- The two follower files have no header and the README says the column order was swapped - the direction (follower vs followed) must be verified from the data.


In [8]:
users_csv = (RAW / "users.csv").as_posix()

con.sql(f"""
    SELECT type, count(*) AS n
    FROM read_csv('{users_csv}',
                  delim = '|',
                  header = false,
                  columns = {{'user_id': 'VARCHAR', 'type': 'VARCHAR', 'name': 'VARCHAR', 'reserved': 'VARCHAR', 'description': 'VARCHAR'}})
    GROUP BY type
    ORDER BY n DESC
""")
        

┌─────────┬────────┐
│  type   │   n    │
│ varchar │ int64  │
├─────────┼────────┤
│ user    │ 625682 │
│ group   │  46158 │
└─────────┴────────┘

### Findings – users vs groups
- `users.csv` has **671,840 accounts**: **625,682 users** and **46,158 groups** (6.9%).
- **Decision:** groups are excluded from the analysis. The research questions are about the behaviour of *individual* users, and groups are communities, not people.
- Unit of analysis = **the individual user**. The raw data is at *event* level (posts, comments, likes) and must be aggregated to one row per user.


In [12]:
con.sql(f"""
    SELECT coalesce(type, 'TOTAL')                        AS type,
           count(*) FILTER (WHERE description <> 'null') AS with_description,
           count(*) FILTER (WHERE description =  'null') AS no_description,
           count(*)                                       AS total,
           round(100.0 * count(*) FILTER (WHERE description <> 'null') / count(*), 1)
                                                          AS pct_with_description
    FROM read_csv('{users_csv}',
                  delim = '|',
                  header = false,
                  columns = {{'user_id': 'VARCHAR', 'type': 'VARCHAR', 'name': 'VARCHAR',
                             'reserved': 'VARCHAR', 'description': 'VARCHAR'}})
    GROUP BY ROLLUP (type)
    ORDER BY total
""")
        

┌─────────┬──────────────────┬────────────────┬────────┬──────────────────────┐
│  type   │ with_description │ no_description │ total  │ pct_with_description │
│ varchar │      int64       │     int64      │ int64  │        double        │
├─────────┼──────────────────┼────────────────┼────────┼──────────────────────┤
│ group   │            31470 │          14688 │  46158 │                 68.2 │
│ user    │           142306 │         483376 │ 625682 │                 22.7 │
│ TOTAL   │           173776 │         498064 │ 671840 │                 25.9 │
└─────────┴──────────────────┴────────────────┴────────┴──────────────────────┘

### Findings – profile descriptions

- Only **22.7% of users** have a description - the attribute is **sparse**.
- The text itself is **not used**: it is mostly missing, multilingual, and would need text mining (outside the course scope - feedback from the teacher).
- Kept at most as a **binary feature `has_description`**, a possible signal of profile effort/engagement.
- Groups describe themselves far more often (68%) than users (23%) - further evidence that groups behave differently and should be excluded.
- Note: the README says about 20% of profiles have a description; the data shows 25.9%. We report what the data shows.


### Saving the clean users table to Parquet

In [64]:
users_parquet = (OUT / "users.parquet").as_posix()

con.sql(f"""
    COPY (
        SELECT user_id,
               any_value(type)                AS type,
               any_value(name)                AS name,
               bool_or(description <> 'null') AS has_description
        FROM read_csv('{users_csv}',
                      delim = '|',
                      header = false,
                      columns = {{'user_id': 'VARCHAR', 'type': 'VARCHAR', 'name': 'VARCHAR',
                                 'reserved': 'VARCHAR', 'description': 'VARCHAR'}})
        GROUP BY user_id
    ) TO '{users_parquet}' (FORMAT parquet)
""")
print("saved:", users_parquet)


saved: c:/Users/HP/Downloads/Data Mining I/Project/data/users.parquet


Checking the saved file is correct

In [66]:
con.sql(f"""
    SELECT count(*) AS n_rows, count(DISTINCT user_id) AS n_unique
    FROM '{users_parquet}'
""")


┌────────┬──────────┐
│ n_rows │ n_unique │
│ int64  │  int64   │
├────────┼──────────┤
│ 651870 │   651870 │
└────────┴──────────┘

In [67]:
con.sql(f"""
    SELECT type,
           count(*)  as n,
        round(100.0 * avg(has_description::INT), 1) AS pct_with_description
    FROM '{users_parquet}'
    GROUP BY type
""")

┌─────────┬────────┬──────────────────────┐
│  type   │   n    │ pct_with_description │
│ varchar │ int64  │        double        │
├─────────┼────────┼──────────────────────┤
│ group   │  44952 │                 68.1 │
│ user    │ 606918 │                 22.4 │
└─────────┴────────┴──────────────────────┘

Checking both CSV and Parquet file size

In [68]:
import os

print(round(os.path.getsize(RAW / "users.csv") / 1e6, 1), "MB (CSV)")
print(round(os.path.getsize(OUT / "users.parquet") / 1e6, 1), "MB (Parquet)")

43.0 MB (CSV)
15.6 MB (Parquet)


### Findings - users table saved
- Saved `data/users.parquet` with columns: `user_id`, `type`, `name`, `has_description` (boolean).
- Dropped: `reserved` (meaningless code) and the description text (replaced by `has_description`).
- Groups are **kept in this file** on purpose: cleaning (format) and selection (what to analyse) are separate steps. Groups are removed in the feature-building step.
- `users.csv` has 671,840 rows but only **651,870 unique accounts**: **19,970 duplicate rows** (11,544 accounts listed twice, 4,213 three times; only 3 differ, by a changed profile name). Deduplicated with `GROUP BY user_id`.
- Unique accounts: **606,918 users** and **44,952 groups** (6.9%).
- File size: CSV 43 MB → Parquet 15.6 MB.


Saving Services table to parquet

In [ ]:
services_parquet = (OUT / "services.parquet").as_posix()

con.sql(f"""
    COPY (
        SELECT DISTINCT user_id,
                        service_id,
                        service_name
        FROM read_csv('{(RAW / "services.csv").as_posix()}',
                       delim = '|',
                       header = false,
                       ignore_errors = true,
                       columns = {{'user_id': 'VARCHAR', 'service_id': 'VARCHAR', 'service_name': 'VARCHAR', 'service_url': 'VARCHAR', 'name_on_service': 'VARCHAR', 'url_on_service': 'VARCHAR'}})
    ) TO '{services_parquet}' (FORMAT PARQUET)
""")

print("saved:", services_parquet)

In [20]:
with open(RAW / "services.csv", "rb") as f:
    raw_lines = sum(1 for _ in f)
print(raw_lines)

1587347


In [21]:
services_csv = (RAW / "services.csv").as_posix()
services_cols = """{'user_id': 'VARCHAR', 'service_id': 'VARCHAR', 'service_name': 'VARCHAR',
                   'service_url': 'VARCHAR', 'name_on_service': 'VARCHAR', 'url_on_service': 'VARCHAR'}"""

rows_read = con.sql(f"""
    SELECT count(*)
    FROM read_csv('{services_csv}', delim='|', header=false, ignore_errors=true,
                  columns={services_cols})
""").fetchone()[0]

print(rows_read, "rows read")
print(raw_lines - rows_read, "lines skipped")


1587275 rows read
72 lines skipped


In [23]:
con.sql(f"""
        SELECT count(*) AS n_rows, count(DISTINCT user_id) AS n_users
        FROM '{services_parquet}'
        """)

┌─────────┬─────────┐
│ n_rows  │ n_users │
│  int64  │  int64  │
├─────────┼─────────┤
│ 1206676 │  322967 │
└─────────┴─────────┘

### Findings - services table
- `services.csv`: **1,587,347 raw lines**, of which **1,587,275 rows were read** - **72 lines skipped (0.005%)**.
  - Measured as: raw line count (Python) − rows read by DuckDB with `ignore_errors = true`.
  - Cause: some values contain the separator `|` (e.g. the Flickr account name `maney|digital`), so the line has too many columns; others contain a line break, which splits one record over two lines.
- **Decision:** skip these lines rather than repair them. Repairing them would be guesswork, and 0.005% of the rows cannot affect association rules mined at a support of about 1%.
- **Duplicates removed** with `SELECT DISTINCT` (1,587,275 → 1,206,676 rows): a user can link several accounts of the same service (e.g. several blogs). For association rules only *whether* a user uses a service matters, not *how many* accounts they have.
- Kept columns: `user_id`, `service_id`, `service_name`. Dropped: service URL and the user's account name/URL on the service (not needed, and the source of the parsing errors).
- **322,967 users** have at least one linked service (52% of the 625,682 users).


### Entries: Definning the files and counting the rows

In [24]:
entry_files = [(RAW / f"entries{i}.csv").as_posix() for i in (1, 2, 3)]

entry_cols = """{'post_id': 'VARCHAR', 'user_id': 'VARCHAR', 'source_name': 'VARCHAR',
                 'source_url': 'VARCHAR', 'geo_x': 'VARCHAR', 'geo_y': 'VARCHAR',
                 'ts': 'VARCHAR', 'text': 'VARCHAR', 'n_img': 'VARCHAR',
                 'img_urls': 'VARCHAR', 'n_vid': 'VARCHAR', 'vid_urls': 'VARCHAR'}"""

entries_src = f"""read_csv({entry_files},
                           delim = '\t', header = false,
                           quote = '', escape = '', nullstr = '\\N',
                           columns = {entry_cols})"""

con.sql(f"SELECT count(*) AS n_rows FROM {entries_src}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┐
│  n_rows  │
│  int64   │
├──────────┤
│ 12450658 │
└──────────┘

### Checking the completeness of each entry columns

In [25]:
con.sql(f"""
        SELECT count(*)   AS n_rows,
            count(geo_x)      AS geo_x,
            count(geo_y)      AS geo_y,
            count(source_name) AS source_name,
            count(*)  FILTER  (WHERE source_name = '') AS source_name_empty,
            count(ts)         AS ts,
            count(n_img)      AS n_img, 
            count(n_vid)      AS n_vid   
        FROM {entries_src}
        """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬───────┬───────┬─────────────┬───────────────────┬──────────┬──────────┬──────────┐
│  n_rows  │ geo_x │ geo_y │ source_name │ source_name_empty │    ts    │  n_img   │  n_vid   │
│  int64   │ int64 │ int64 │    int64    │       int64       │  int64   │  int64   │  int64   │
├──────────┼───────┼───────┼─────────────┼───────────────────┼──────────┼──────────┼──────────┤
│ 12450658 │     0 │     0 │    12450658 │           1036763 │ 12450658 │ 12450658 │ 12450658 │
└──────────┴───────┴───────┴─────────────┴───────────────────┴──────────┴──────────┴──────────┘

### Cleaning the Entries table and save what we need

In [28]:
entries_parquet = (OUT / "entries.parquet").as_posix()

con.sql(f"""
        COPY (
            SELECT  post_id,
                    user_id,
                    source_name,
                    CAST( ts AS TIMESTAMP) AS ts,
                    length(text)           AS text_len,
                    text LIKE '%<a %'      AS has_link,
                    CAST(n_img AS INTEGER) AS n_img,
                    CAST(n_vid AS INTEGER) AS n_vid
            FROM {entries_src}
        ) TO '{entries_parquet}' (FORMAT PARQUET)

        """)

print("Saved:", entries_parquet)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: c:/Users/HP/Downloads/Data Mining I/Project/data/entries.parquet


### Checking the saved file

In [30]:
con.sql(f"""
        SELECT  count(*)                                    AS n_posts, 
                count(DISTINCT post_id)                     AS n_distinct_posts,
                count(DISTINCT user_id)                     AS n_authors,
                min(ts)                                     AS first_ts,
                max(ts)                                     AS last_ts,
                count(*) FILTER (WHERE ts < '2010-08-01')   AS before_aug,
                round(100 * avg((n_img > 0)::INT), 1)       AS pct_with_image,
                round(100 * avg((n_vid > 0)::INT), 1)       AS pct_with_video,
                round(100 * avg(has_link::INT), 1)          AS pct_with_link
    FROM '{entries_parquet}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬──────────────────┬───────────┬─────────────────────┬─────────────────────┬────────────┬────────────────┬────────────────┬───────────────┐
│ n_posts  │ n_distinct_posts │ n_authors │      first_ts       │       last_ts       │ before_aug │ pct_with_image │ pct_with_video │ pct_with_link │
│  int64   │      int64       │   int64   │      timestamp      │      timestamp      │   int64    │     double     │     double     │    double     │
├──────────┼──────────────────┼───────────┼─────────────────────┼─────────────────────┼────────────┼────────────────┼────────────────┼───────────────┤
│ 12450658 │         12450658 │    256544 │ 1970-01-01 00:00:01 │ 2010-09-30 23:59:59 │      13282 │            7.8 │            1.5 │          86.9 │
└──────────┴──────────────────┴───────────┴─────────────────────┴─────────────────────┴────────────┴────────────────┴────────────────┴───────────────┘

# Checking the file size of the raw CSV and parquet file

In [31]:
print(round(sum(os.path.getsize(f) for f in entry_files) / 1e9, 2), "GB  (3 CSVs)")
print(round(os.path.getsize(entries_parquet) / 1e6, 1), "MB  (Parquet)")

4.66 GB  (3 CSVs)
632.5 MB  (Parquet)


### Findings – entries (posts)
- 3 files, **12,450,658 posts**, all lines parsed (row count = line count). `post_id` is unique - no duplicate posts.
- **256,544 distinct authors** - far fewer than the 625,682 registered users → most users never posted in the window.
- Quoting disabled explicitly (`quote=''`): post texts contain `"` characters that are not CSV quotes; auto-detection only samples the first 20,480 lines.
- **Missing values:** GeoX/GeoY are 100% empty → dropped. `source_name` is never NULL but **1,036,763 posts (8.3%) have an empty source** = posted directly on FriendFeed (native), a meaningful value.
- **Timestamps:** 13,282 posts (0.1%) are dated before the monitoring period (1 Aug 2010), including 24 at 1970-01-01 (Unix epoch = missing date). These will be removed when restricting to the study window.
- **Text dropped** (scope: no text mining). Kept two language-independent summaries: `text_len` and `has_link`.
- **Content is imbalanced:** 7.8% of posts have images, 1.5% videos, 86.9% links - rules about image/video posts will have low support.
- Size: 4.66 GB CSV - ~632 MB Parquet (text column dropped + compression).


### Comment: Definning the source and counting the rows

In [37]:
comment_cols = """{'comment_id': 'VARCHAR', 'post_id': 'VARCHAR', 'user_id': 'VARCHAR',
                   'source_name': 'VARCHAR', 'source_url': 'VARCHAR', 'geo_x': 'VARCHAR',
                   'geo_y': 'VARCHAR', 'ts': 'VARCHAR', 'text': 'VARCHAR'}"""

comments_src = f"""read_csv('{(RAW / "commentAugSept.csv").as_posix()}',
                            delim = '\t', header = false,
                            quote = '', escape = '', nullstr = '\\N',
                            columns = {comment_cols})"""

con.sql(f"SELECT count(*) AS n_rows FROM {comments_src}")



┌─────────┐
│ n_rows  │
│  int64  │
├─────────┤
│ 3749891 │
└─────────┘

### Cleaning the comment table and save what we need

In [35]:
comments_parquet = (OUT / "comments.parquet").as_posix()

con.sql(f"""
    COPY (
        SELECT comment_id,
               post_id,
               user_id,
               source_name,
               CAST(ts AS TIMESTAMP) AS ts,
               length(text)           AS text_len
        FROM {comments_src}
    ) TO '{comments_parquet}' (FORMAT parquet)
""")

print("saved:", comments_parquet)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

saved: c:/Users/HP/Downloads/Data Mining I/Project/data/comments.parquet


### Checking the saved file

In [ ]:
con.sql(f"""
        SELECT  count(*)                                    AS n_comments, 
                count(DISTINCT comment_id)                  AS n_distinct_comments,
                count(DISTINCT user_id)                     AS n_commenters,
                min(ts)                                     AS first_ts,
                max(ts)                                     AS last_ts,
                count(*) FILTER (WHERE ts < '2010-08-01')   AS before_aug,
                count(*) FILTER (WHERE source_name = '')    AS n_empty_comments
    FROM '{comments_parquet}'
""")

┌────────────┬─────────────────────┬──────────────┬─────────────────────┬─────────────────────┬────────────┬──────────────────┐
│ n_comments │ n_distinct_comments │ n_commenters │      first_ts       │       last_ts       │ before_aug │ n_empty_comments │
│   int64    │        int64        │    int64     │      timestamp      │      timestamp      │   int64    │      int64       │
├────────────┼─────────────────────┼──────────────┼─────────────────────┼─────────────────────┼────────────┼──────────────────┤
│    3749891 │             3749891 │        54045 │ 1970-01-01 00:00:01 │ 2010-09-30 23:59:58 │     130995 │          3348335 │
└────────────┴─────────────────────┴──────────────┴─────────────────────┴─────────────────────┴────────────┴──────────────────┘

### Orphan comments

In [39]:
con.sql(f"""
    SELECT count(*)                                  AS n_comments,
           count(*) FILTER (WHERE e.post_id IS NULL) AS orphan_comments
    FROM '{comments_parquet}' AS c
    LEFT JOIN (SELECT post_id FROM '{entries_parquet}') AS e
           ON c.post_id = e.post_id
""")


┌────────────┬─────────────────┐
│ n_comments │ orphan_comments │
│   int64    │      int64      │
├────────────┼─────────────────┤
│    3749891 │             217 │
└────────────┴─────────────────┘

### Findings - comments
- **3,749,891 comments**, all lines parsed; `comment_id` is unique - no duplicates.
- **Documentation ≠ data:** the file has 9 columns, not 13 as in the README (no image/video columns). Column names are assigned by position, so the order must match the file exactly (a wrong order made `ts` point at the empty `source_url` column - caught by the failing `CAST`).
- **Only 54,045 users wrote comments** (8.6% of users, ~21% of posters) - conversation is concentrated in a small minority; comment features will be 0 for most users (very skewed distributions).
- **89% of comments are native** (empty source) vs 8% of posts - posts are often pushed in automatically by external tools, while comments are mostly typed by people on FriendFeed. Comment activity is therefore a strong signal of human interaction.
- **Timestamps:** 130,995 comments (3.5%) are dated before the monitoring period (1 Aug 2010), incl. 1970-01-01 (Unix epoch = missing date) - removed when restricting to the study window.
- **Referential integrity:** 217 comments (0.006%) point to a post not in the entries data (probably posted before monitoring began). They can count as comments *written*, but not as comments *received*, because the post's author is unknown.
- Text dropped; kept `text_len` only.


### Likes: Definning the source and looking at the data

In [40]:
likes_cols = """{'user_id': 'VARCHAR', 'post_id': 'VARCHAR', 'ts': 'VARCHAR'}"""

likes_src = f"""read_csv('{(RAW / "likes.csv").as_posix()}',
                            delim = '\t', header = false,
                            quote = '', escape = '', nullstr = '\\N',
                            columns = {likes_cols})"""


con.sql(f"""
    SELECT count(*)                          AS n_likes,
           count(DISTINCT (user_id, post_id)) AS n_distinct_pairs,
           count(DISTINCT user_id)           AS n_likers,
           count(DISTINCT post_id)           AS n_liked_posts,
           min(CAST(ts AS TIMESTAMP))        AS first_ts,
           max(CAST(ts AS TIMESTAMP))        AS last_ts
    FROM {likes_src}
""")


┌─────────┬──────────────────┬──────────┬───────────────┬─────────────────────┬─────────────────────┐
│ n_likes │ n_distinct_pairs │ n_likers │ n_liked_posts │      first_ts       │       last_ts       │
│  int64  │      int64       │  int64   │     int64     │      timestamp      │      timestamp      │
├─────────┼──────────────────┼──────────┼───────────────┼─────────────────────┼─────────────────────┤
│  798112 │           797290 │    22876 │        475596 │ 2008-03-05 04:36:07 │ 2010-10-16 21:42:49 │
└─────────┴──────────────────┴──────────┴───────────────┴─────────────────────┴─────────────────────┘

### Cleaning the likes table and save what we need

In [41]:
likes_parquet = (OUT / "likes.parquet").as_posix()

con.sql(f"""
    COPY (
        SELECT user_id,
               post_id,
               min(CAST(ts AS TIMESTAMP)) AS ts
        FROM {likes_src}
        GROUP BY user_id, post_id
    ) TO '{likes_parquet}' (FORMAT parquet)
""")
print("saved:", likes_parquet)

saved: c:/Users/HP/Downloads/Data Mining I/Project/data/likes.parquet


### Checking the saved file and the time window

In [42]:
con.sql(f"""
    SELECT count(*)                                                        AS n_likes,
           count(*) FILTER (WHERE ts <  '2010-08-01')                      AS before_window,
           count(*) FILTER (WHERE ts >= '2010-10-01')                      AS after_window,
           count(*) FILTER (WHERE ts >= '2010-08-01' AND ts < '2010-10-01') AS in_window
    FROM '{likes_parquet}'
""")


┌─────────┬───────────────┬──────────────┬───────────┐
│ n_likes │ before_window │ after_window │ in_window │
│  int64  │     int64     │    int64     │   int64   │
├─────────┼───────────────┼──────────────┼───────────┤
│  797290 │        125550 │       131408 │    540332 │
└─────────┴───────────────┴──────────────┴───────────┘

### Orphan check

In [45]:
con.sql(f"""
    WITH likes_with_period AS (
        SELECT l.post_id,
               e.post_id AS matched_post,
               CASE
                   WHEN l.ts <  '2010-08-01' THEN '1 before window'
                   WHEN l.ts >= '2010-10-01' THEN '3 after window'
                   ELSE                           '2 in window'
               END AS period
        FROM '{likes_parquet}' AS l
        LEFT JOIN (SELECT post_id FROM '{entries_parquet}') AS e
               ON l.post_id = e.post_id
    )
    SELECT coalesce(period, 'TOTAL')                          AS period,
           count(*)                                           AS n_likes,
           count(*) FILTER (WHERE matched_post IS NULL)       AS orphan_likes,
           round(100.0 * count(*) FILTER (WHERE matched_post IS NULL) / count(*), 1)
                                                              AS pct_orphan
    FROM likes_with_period
    GROUP BY ROLLUP (likes_with_period.period)
    ORDER BY likes_with_period.period NULLS LAST
""")

┌─────────────────┬─────────┬──────────────┬────────────┐
│     period      │ n_likes │ orphan_likes │ pct_orphan │
│     varchar     │  int64  │    int64     │   double   │
├─────────────────┼─────────┼──────────────┼────────────┤
│ 1 before window │  125550 │         8941 │        7.1 │
│ 2 in window     │  540332 │          442 │        0.1 │
│ 3 after window  │  131408 │       129204 │       98.3 │
│ TOTAL           │  797290 │       138587 │       17.4 │
└─────────────────┴─────────┴──────────────┴────────────┘

### Findings - likes
- **798,112 like rows; 822 duplicates** (same user liking the same post twice - impossible on FriendFeed, so a recording error); deduplicated with `GROUP BY user_id, post_id`, keeping the earliest timestamp: **797,290 likes**.
- **Only 22,876 users ever liked a post** (3.7% of users); like-based features are 0 for most users.
- **Dates range from 5 Mar 2008 to 16 Oct 2010**: 125,550 likes before and 131,408 after the Aug–Sep window; **540,332 likes in the study period**.
- **17.4% of likes are orphans** (post not in entries), but this is a time-coverage issue, not an error: 98.3% of likes given *after* 30 Sep point to posts written in October, which were never collected. **Within the Aug–Sep window only 442 likes (0.1%) are orphans** - restricting all tables to the same window solves it.


### Subscription: Definning the source and looking at the data

In [50]:
subs_src = f"""read_csv('{(RAW / "subscriptions.csv").as_posix()}',
                        delim = ',', header = false, quote = '', escape = '',
                        columns = {{'col1': 'VARCHAR', 'col2': 'VARCHAR'}})"""

print("Most frequent names in column 1:")
print(con.sql(f"SELECT col1, count(*) AS n FROM {subs_src} GROUP BY col1 ORDER BY n DESC LIMIT 5"))

print("Most frequent names in column 2:")
print(con.sql(f"SELECT col2, count(*) AS n FROM {subs_src} GROUP BY col2 ORDER BY n DESC LIMIT 5"))


Most frequent names in column 1:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬────────┐
│     col1     │   n    │
│   varchar    │ int64  │
├──────────────┼────────┤
│ br3adman     │ 112145 │
│ malikimran   │ 109146 │
│ malikimrana1 │ 104794 │
│ phasuk111    │  99950 │
│ sonergonul   │  98793 │
└──────────────┴────────┘

Most frequent names in column 2:
┌─────────────┬────────┐
│    col2     │   n    │
│   varchar   │ int64  │
├─────────────┼────────┤
│ barackobama │ 113923 │
│ mashable    │  87461 │
│ leolaporte  │  77467 │
│ scobleizer  │  51043 │
│ kevinrose   │  31959 │
└─────────────┴────────┘



### Saving the subscription table with the correct column name as follows

In [47]:
follows_parquet = (OUT / "follows.parquet").as_posix()

con.sql(f"""
    COPY (
        SELECT DISTINCT col1 AS follower,
                        col2 AS followed
        FROM {subs_src}
    ) TO '{follows_parquet}' (FORMAT parquet)
""")
print("saved:", follows_parquet)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

saved: c:/Users/HP/Downloads/Data Mining I/Project/data/follows.parquet


In [48]:
with open(RAW / "subscriptions.csv", "rb") as f:
    subs_lines = sum(1 for _ in f)
print(subs_lines, "raw lines")


27811816 raw lines


### Checking the saved file

In [51]:
con.sql(f"""
    SELECT count(*)                                    AS n_edges,
           {subs_lines} - count(*)                     AS duplicates_removed,
           count(DISTINCT follower)                    AS n_followers,
           count(DISTINCT followed)                    AS n_followed,
           count(*) FILTER (WHERE follower = followed) AS self_follows
    FROM '{follows_parquet}'
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬────────────────────┬─────────────┬────────────┬──────────────┐
│ n_edges  │ duplicates_removed │ n_followers │ n_followed │ self_follows │
│  int64   │       int64        │    int64    │   int64    │    int64     │
├──────────┼────────────────────┼─────────────┼────────────┼──────────────┤
│ 25862827 │            1948989 │      535673 │     536131 │            0 │
└──────────┴────────────────────┴─────────────┴────────────┴──────────────┘

### Checking how unequal the network is?

In [52]:
con.sql(f"""
    SELECT max(n)    AS max_followers,
           median(n) AS median_followers,
           round(avg(n), 1) AS mean_followers
    FROM (SELECT followed, count(*) AS n
          FROM '{follows_parquet}'
          GROUP BY followed)
""")

┌───────────────┬──────────────────┬────────────────┐
│ max_followers │ median_followers │ mean_followers │
│     int64     │      double      │     double     │
├───────────────┼──────────────────┼────────────────┤
│         51043 │              5.0 │           48.2 │
└───────────────┴──────────────────┴────────────────┘

### Checking the most followed accounts after dupicates removed

In [57]:
con.sql(f"""
    WITH raw_counts AS (
        SELECT col2 AS account, count(*) AS raw_rows
        FROM {subs_src}
        GROUP BY col2
    ),
    unique_counts AS (
        SELECT followed AS account, count(*) AS unique_followers
        FROM '{follows_parquet}'
        GROUP BY followed
    )
    SELECT u.account,
           u.unique_followers,
           r.raw_rows
    FROM unique_counts AS u
    JOIN raw_counts    AS r ON u.account = r.account
    ORDER BY u.unique_followers DESC
    LIMIT 5
""")


┌─────────────┬──────────────────┬──────────┐
│   account   │ unique_followers │ raw_rows │
│   varchar   │      int64       │  int64   │
├─────────────┼──────────────────┼──────────┤
│ scobleizer  │            51043 │    51043 │
│ mashable    │            43733 │    87461 │
│ leolaporte  │            38735 │    77467 │
│ barackobama │            37980 │   113923 │
│ kevinrose   │            31959 │    31959 │
└─────────────┴──────────────────┴──────────┘

### Findings - follower network (subscriptions.csv, snapshot 15–16 Oct 2010)
- **The README does not match the files:** it documents one "Following" file (static, columns FollowedID, FollowerID), but the dataset contains two follower files: `subscriptions.csv` (2 columns, no timestamp - not mentioned in the README) and `followingAugSept.csv` (3 columns incl. a timestamp).
- **Direction verified from the data:** in `subscriptions.csv`, column 2 contains well-known accounts (barackobama, mashable, leolaporte, scobleizer, kevinrose); **column 1 = follower, column 2 = followed**, the *opposite* of the README's order. Trusting the README would have swapped every user's follower and following counts.
- 27,811,816 lines, all read (0 skipped) - **25,862,827 unique edges; 1,948,989 duplicates removed (7%)**. No self-follows. A follow relationship either exists or not, so a repeated edge can only be a recording error.
- **Duplicates changed the ranking of the most-followed accounts:** barackobama had 113,923 raw rows but only 37,980 unique followers; mashable 87,461 - 43,733; leolaporte 77,467 - 38,735. After deduplication the top accounts are: scobleizer 51,043 · mashable 43,733 · leolaporte 38,735 · barackobama 37,980 · kevinrose 31,959. Raw counts would have misrepresented exactly the most influential accounts.
- **Heavy-tailed:** median 5 followers, mean 48.2, max 51,043 - the mean is ~10× the median, driven by a few huge accounts - log transform / robust scaling needed before clustering.
- **Follow-spam accounts:** some users follow more than 90,000 unique accounts (br3adman 105,618; malikimran 102,782; malikimrana1 98,711) - likely automated; relevant for the bot archetype.
- **Limitation:** the snapshot is from mid-October, slightly after the Aug–Sep activity window.
- Use: in-degree (followers) and out-degree (following) per user as features - no further graph analysis (scope agreed with the teacher).


### followingAugSep: Definning the source and looking at the source

In [58]:
events_src = f"""read_csv('{(RAW / "followingAugSept.csv").as_posix()}',
                          delim = '\t', header = false, quote = '', escape = '',
                          columns = {{'col1': 'VARCHAR', 'col2': 'VARCHAR', 'ts': 'VARCHAR'}})"""

print(con.sql(f"""
    SELECT count(*)                     AS n_rows,
           count(DISTINCT (col1, col2)) AS unique_edges,
           min(CAST(ts AS TIMESTAMP))   AS first_ts,
           max(CAST(ts AS TIMESTAMP))   AS last_ts
    FROM {events_src}
"""))

print("Most frequent in column 1:")
print(con.sql(f"SELECT col1, count(*) AS n FROM {events_src} GROUP BY col1 ORDER BY n DESC LIMIT 5"))
print("Most frequent in column 2:")
print(con.sql(f"SELECT col2, count(*) AS n FROM {events_src} GROUP BY col2 ORDER BY n DESC LIMIT 5"))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬──────────────┬────────────────────────────┬────────────────────────────┐
│  n_rows  │ unique_edges │          first_ts          │          last_ts           │
│  int64   │    int64     │         timestamp          │         timestamp          │
├──────────┼──────────────┼────────────────────────────┼────────────────────────────┤
│ 19547158 │     19547158 │ 2010-08-01 16:21:37.562726 │ 2010-09-30 23:59:12.242897 │
└──────────┴──────────────┴────────────────────────────┴────────────────────────────┘

Most frequent in column 1:
┌─────────────────┬───────┐
│      col1       │   n   │
│     varchar     │ int64 │
├─────────────────┼───────┤
│ mashable        │ 43222 │
│ leolaporte      │ 38260 │
│ barackobama     │ 37400 │
│ kevinrose       │ 31696 │
│ veronicabelmont │ 29441 │
└─────────────────┴───────┘

Most frequent in column 2:
┌──────────────┬───────┐
│     col2     │   n   │
│   varchar    │ int64 │
├──────────────┼───────┤
│ br3adman     │ 74522 │
│ malikimran   │ 72064 

### Checking when do the "follow events" happen?

In [60]:
con.sql(f"""
    SELECT date_diff('day', DATE '2010-08-01', CAST(ts AS TIMESTAMP)) // 7 + 1 AS study_week,
           min(CAST(ts AS DATE))                                           AS first_day,
           max(CAST(ts AS DATE))                                           AS last_day,
           count(*)                                                        AS n_events
    FROM {events_src}
    GROUP BY study_week
    ORDER BY study_week
""")


┌────────────┬────────────┬────────────┬──────────┐
│ study_week │ first_day  │  last_day  │ n_events │
│   int64    │    date    │    date    │  int64   │
├────────────┼────────────┼────────────┼──────────┤
│          1 │ 2010-08-01 │ 2010-08-07 │ 11211923 │
│          2 │ 2010-08-08 │ 2010-08-14 │  3566253 │
│          3 │ 2010-08-15 │ 2010-08-21 │  1127810 │
│          4 │ 2010-08-22 │ 2010-08-28 │  1520277 │
│          5 │ 2010-08-29 │ 2010-09-04 │   771043 │
│          6 │ 2010-09-05 │ 2010-09-11 │   446491 │
│          7 │ 2010-09-12 │ 2010-09-18 │   379820 │
│          8 │ 2010-09-19 │ 2010-09-25 │   325693 │
│          9 │ 2010-09-26 │ 2010-09-30 │   197848 │
└────────────┴────────────┴────────────┴──────────┘

### Are these new relationships?

In [63]:
con.sql(f"""
    SELECT count(*)                                       AS n_events,
           count(*) FILTER (WHERE f.follower IS NOT NULL) AS also_in_subscriptions,
           round(100.0 * count(*) FILTER (WHERE f.follower IS NOT NULL) / count(*), 1) AS pct
    FROM {events_src} AS ev
    LEFT JOIN '{follows_parquet}' AS f
           ON  ev.col2 = f.follower      
           AND ev.col1 = f.followed      
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬───────────────────────┬────────┐
│ n_events │ also_in_subscriptions │  pct   │
│  int64   │         int64         │ double │
├──────────┼───────────────────────┼────────┤
│ 19547158 │              19209576 │   98.3 │
└──────────┴───────────────────────┴────────┘

### Findings - followingAugSept.csv (not used)
- 19,547,158 rows, no duplicates, timestamps 1 Aug 2010 16:21 - 30 Sep 2010.
- **Celebrity test:** famous accounts (mashable, leolaporte, barackobama) are in column 1 - **column 1 = followed, column 2 = follower**; the *opposite* order of `subscriptions.csv` (and the order stated in the README).
- **The timestamps do not record when users started following:** 11.2M rows (57%) fall in the first study week (1 - 7 Aug), then decline steadily (week 2: 3.6M … week 8: 0.3M). This is the pattern of a crawler first recording the whole existing network and then only the few relationships it had not seen before - not of real follow behaviour. (Inference from the data; the README does not explain the timestamps.)
- **98.3% of these edges also appear in the October snapshot** (`subscriptions.csv`).
- **Decision:** the file cannot measure "followers gained during the study", so it is **not used**. Follower/following counts are taken from the larger static snapshot.
